# Silver Layer

### Load delta tables

In [0]:
bronze_orders_df=spark.read.table('zomato_project.bronze.bronze_orders')
bronze_customers_df=spark.read.table('zomato_project.bronze.bronze_customers')
bronze_restaurants_df=spark.read.table('zomato_project.bronze.bronze_restaurants')
bronze_deliveries_df=spark.read.table('zomato_project.bronze.bronze_deliveries')
bronze_delivery_persons_df=spark.read.table('zomato_project.bronze.bronze_delivery_persons')

In [0]:
bronze_customers_df.printSchema()
bronze_orders_df.printSchema()
bronze_restaurants_df.printSchema()
bronze_deliveries_df.printSchema()
bronze_delivery_persons_df.printSchema()

### Handling messy data

In [0]:
from pyspark.sql.functions import to_date,col

bronze_customers_df = bronze_customers_df \
    .withColumn("customer_id", col("customer_id").cast("string")) \
    .withColumn("phone", col("phone").cast("string")) \
    .withColumn("signup_date", col("signup_date").cast("string")) \
    .withColumn("total_orders", col("total_orders").cast("string")) \
    .withColumn("average_rating", col("average_rating").cast("string"))

In [0]:
bronze_customers_df.printSchema()

In [0]:
bronze_customers_df = bronze_customers_df.dropna(subset=["customer_id"])
bronze_customers_df = bronze_customers_df.drop_duplicates()
from pyspark.sql.functions import col, when, lower, regexp_replace, concat, lit

from pyspark.sql.functions import col, when, lower, regexp_replace, concat, lit, trim,to_timestamp

bronze_customers_df = bronze_customers_df.withColumn(
    "email",
    when(
        col("email").isNull() | 
        (trim(col("email")) == "") | 
        (col("email").isin("null", "None", "NULL")),
        concat(
            regexp_replace(lower(col("name")), "[^a-z0-9]", ""),
            lit("@example.com")
        )
    ).otherwise(col("email"))
)

bronze_customers_df = bronze_customers_df.fillna({
    "name": "Unknown"
})
from pyspark.sql.functions import col, when, split, trim, lower

bronze_customers_df = bronze_customers_df.withColumn(
    "name",
    when(
        (
            col("name").isNull() |
            (trim(col("name")) == "") |
            (lower(col("name")).isin("null", "none", "unknown"))
        ) & col("email").isNotNull(),
        split(col("email"), "@")[0]
    ).otherwise(col("name"))
)
from pyspark.sql.functions import col, regexp_replace, initcap

bronze_customers_df = bronze_customers_df.withColumn(
    "name",
    initcap(  # Capitalize first letter
        regexp_replace(col("name"), "[0-9]", "")  # remove numbers
    )
)
from pyspark.sql.functions import try_to_date, col , coalesce

bronze_customers_df = bronze_customers_df.withColumn(
    "signup_date",
    coalesce(
     try_to_date(col("signup_date"), "yyyy-MM-dd"),
     try_to_date(col("signup_date"), "dd-MM-yyyy"),
     try_to_date(col("signup_date"), "yyyy/MM/dd"),
     try_to_date(col("signup_date"), "dd/MM/yyyy")
    )
)
from pyspark.sql.functions import col, when, lower, trim

from pyspark.sql.functions import col, coalesce, lit

bronze_customers_df = bronze_customers_df.withColumn(
    "signup_date",
    coalesce(
        col("signup_date").cast("date"),
        lit("2000-01-01").cast("date")
    )
)
from pyspark.sql.functions import col, when, lower, trim

bronze_customers_df = bronze_customers_df.withColumn(
    "phone",
    when(
        col("phone").isNull() |
        (trim(col("phone")) == "") |
        lower(col("phone")).isin("null", "none"),
        None
    ).otherwise(col("phone"))
)
bronze_customers_df=bronze_customers_df.fillna({'phone':'unknown'})
from pyspark.sql.functions import col, when, lower, trim

bronze_customers_df = bronze_customers_df.withColumn(
    "is_premium",
    when(
        col("is_premium").isNull() |
        (trim(col("is_premium")) == "") |
        lower(col("is_premium")).isin("null", "none"),
        "0"
    ).otherwise(trim(col("is_premium")))
)
from pyspark.sql.functions import col, when, lower, trim

bronze_customers_df = bronze_customers_df \
    .withColumn(
        "total_orders",
        when(
            col("total_orders").isNull() |
            lower(trim(col("total_orders"))).isin("null", "none", "", "invalid"),
            None
        ).otherwise(col("total_orders"))
    ) \
    .withColumn(
        "average_rating",
        when(
            col("average_rating").isNull() |
            lower(trim(col("average_rating"))).isin("null", "none", "", "invalid"),
            None
        ).otherwise(col("average_rating"))
    )
bronze_customers_df = bronze_customers_df \
    .withColumn("total_orders", col("total_orders").cast("int")) \
    .withColumn("average_rating", col("average_rating").cast("double"))   
from pyspark.sql.functions import abs

bronze_customers_df = bronze_customers_df \
    .withColumn("total_orders", abs(col("total_orders"))) \
    .withColumn("average_rating", abs(col("average_rating")))
bronze_customers_df = bronze_customers_df.filter(
    (col("total_orders") >= 0) &
    (col("average_rating") >= 0)
)    



from pyspark.sql.functions import col, regexp_replace

bronze_customers_df = bronze_customers_df.withColumn(
    "phone",
    regexp_replace(col("phone"), "[^0-9]", "")
)
from pyspark.sql.functions import when, length, substring

bronze_customers_df = bronze_customers_df.withColumn(
    "phone",
    when(length(col("phone")) == 12, substring(col("phone"), 3, 10))
    .otherwise(col("phone"))
)
bronze_customers_df = bronze_customers_df.withColumn(
    "phone",
    when(length(col("phone")) == 10, col("phone"))
    .otherwise(None)
)
bronze_customers_df=bronze_customers_df.fillna({'phone':'unknown'})

bronze_customers_df = bronze_customers_df.withColumn(
    "customer_id",
    col("customer_id").cast("int")
)
from pyspark.sql.functions import col, when, lower, trim

bronze_customers_df = bronze_customers_df.withColumn(
    "preferred_cuisine",
    when(
        col("preferred_cuisine").isNull() |
        (trim(col("preferred_cuisine")) == "") |
        lower(col("preferred_cuisine")).isin("null", "none"),
        "unknown"
    ).otherwise(trim(col("preferred_cuisine")))
)

bronze_customers_df.display()

In [0]:
from pyspark.sql.functions import regexp_replace, col,row_number
from pyspark.sql.window import Window

window_spec = Window.orderBy(
    regexp_replace(col("restaurant_id"), "[^0-9]", "").cast("int")
)

bronze_restaurants_df = bronze_restaurants_df.withColumn(
    "restaurant_id",
    row_number().over(window_spec)
)

In [0]:
bronze_restaurants_df = bronze_restaurants_df.orderBy("restaurant_id")



from pyspark.sql.functions import (
    col, when, lower, trim, regexp_replace, to_date,
    coalesce, lit, length, initcap
)

# -------------------------------
# 1. Drop critical nulls + duplicates
# -------------------------------
bronze_restaurants_df = bronze_restaurants_df.dropna(subset=["restaurant_id"])
bronze_restaurants_df = bronze_restaurants_df.drop_duplicates()

# -------------------------------
# 2. Clean string nulls (generic)
# -------------------------------
def clean_null(col_name):
    return when(
        col(col_name).isNull() |
        (trim(col(col_name)) == "") |
        lower(col(col_name)).isin("null", "none"),
        None
    ).otherwise(trim(col(col_name)))

columns_to_clean = [
    "name", "cuisine_type", "location", "owner_name",
    "average_delivery_time", "contact_number",
    "rating", "total_orders", "is_active", "opening_date"
]

for c in columns_to_clean:
    bronze_restaurants_df = bronze_restaurants_df.withColumn(c, clean_null(c))

# -------------------------------
# 3. NAME & OWNER CLEANING
# -------------------------------
bronze_restaurants_df = bronze_restaurants_df \
    .withColumn("name", initcap(col("name"))) \
    .withColumn("owner_name", initcap(col("owner_name")))

# -------------------------------
# 4. CONTACT NUMBER CLEANING
# -------------------------------
bronze_restaurants_df = bronze_restaurants_df \
    .withColumn("contact_number", regexp_replace(col("contact_number"), "[^0-9]", "")) \
    .withColumn(
        "contact_number",
        when(length(col("contact_number")) == 12,
             col("contact_number").substr(3, 10))
        .otherwise(col("contact_number"))
    ) \
    .withColumn(
        "contact_number",
        when(length(col("contact_number")) == 10, col("contact_number"))
        .otherwise(None)
    ) \
    .fillna({"contact_number": "unknown"})

# -------------------------------
# 5. DELIVERY TIME → INTEGER
# -------------------------------
bronze_restaurants_df = bronze_restaurants_df.withColumn(
    "average_delivery_time",
    regexp_replace(col("average_delivery_time"), "[^0-9]", "").cast("int")
)

# -------------------------------
# 6. RATING → DOUBLE (0–5)
# -------------------------------
bronze_restaurants_df = bronze_restaurants_df.withColumn(
    "rating",
    col("rating").cast("double")
)

bronze_restaurants_df = bronze_restaurants_df.withColumn(
    "rating",
    when((col("rating") < 0) | (col("rating") > 5), None)
    .otherwise(col("rating"))
)

# -------------------------------
# 7. TOTAL ORDERS → INT
# -------------------------------
bronze_restaurants_df = bronze_restaurants_df.withColumn(
    "total_orders",
    col("total_orders").cast("int")
)

bronze_restaurants_df = bronze_restaurants_df.fillna({
    "total_orders": 0
})
bronze_restaurants_df=bronze_restaurants_df.withColumn("total_orders",when(col("total_orders")<0,abs(col("total_orders"))).otherwise(col("total_orders")))
# -------------------------------
# 8. IS_ACTIVE → STRING CLEAN
# -------------------------------
bronze_restaurants_df = bronze_restaurants_df.withColumn(
    "is_active",
    when(col("is_active").isin("1", "0"), col("is_active"))
    .otherwise(None)
)

# -------------------------------
# 9. OPENING DATE → DATE
# -------------------------------

from pyspark.sql.functions import try_to_date, coalesce, col

bronze_restaurants_df = bronze_restaurants_df.withColumn(
    "opening_date",
    coalesce(
        try_to_date(col("opening_date"), "yyyy-MM-dd"),
        try_to_date(col("opening_date"), "dd-MM-yyyy"),
        try_to_date(col("opening_date"), "yyyy/MM/dd"),
        try_to_date(col("opening_date"), "dd/MM/yyyy")
    )
)
from pyspark.sql.functions import lit

bronze_restaurants_df = bronze_restaurants_df.withColumn(
    "opening_date",
    coalesce(
        col("opening_date"),
        lit("2000-01-01").cast("date")
    )
)


# -------------------------------
# 10. CUISINE STANDARDIZATION
# -------------------------------
bronze_restaurants_df = bronze_restaurants_df.withColumn(
    "cuisine_type",
    when(col("cuisine_type").isNull(), "unknown")
    .otherwise(initcap(col("cuisine_type")))
)
bronze_restaurants_df = bronze_restaurants_df.orderBy("restaurant_id")

bronze_restaurants_df=bronze_restaurants_df.drop("opening_date")

bronze_restaurants_df.display()

In [0]:
from pyspark.sql.functions import to_date

bronze_customers_df = bronze_customers_df \
    .withColumn("customer_id", col("customer_id").cast("int")) \
    .withColumn("phone", col("phone").cast("string")) \
    .withColumn("signup_date", to_date(col("signup_date"), "yyyy-MM-dd")) \
    .withColumn("total_orders", col("total_orders").cast("int")) \
    .withColumn("average_rating", col("average_rating").cast("double"))

In [0]:
bronze_orders_df=bronze_orders_df.withColumn("order_id",col("order_id").cast("int"))\
    .withColumn("customer_id",col("customer_id").cast("int"))\
    .withColumn("restaurant_id",col("restaurant_id").cast("int"))\
    .withColumn("status",col("status").cast("string"))\
    .withColumn("total_amount",col("total_amount").cast("double"))\
    .withColumn("payment_mode",col("payment_mode").cast("string"))\
    .withColumn("discount_applied",col("discount_applied").cast("double"))\
    .withColumn("feedback_rating",col("feedback_rating").cast("double"))

In [0]:
bronze_orders_df.printSchema()

In [0]:
from pyspark.sql.functions import expr

# convert string "null" → real null
bronze_orders_df = bronze_orders_df.replace(
    ["null", "NULL", ""], None
)

# cast numeric columns safely
bronze_orders_df = bronze_orders_df.withColumn(
    "total_amount", expr("try_cast(total_amount as double)")
).withColumn(
    "discount_applied", expr("try_cast(discount_applied as double)")
).withColumn(
    "feedback_rating", expr("try_cast(feedback_rating as double)")
)

In [0]:
from pyspark.sql.functions import abs, col, when, to_timestamp, expr, trim

# Step 1: Fill nulls
bronze_orders_df = bronze_orders_df.fillna({
    "status": "Unknown",
    "payment_mode": "Unknown",
    "total_amount": 0,
    "discount_applied": 0,
    "feedback_rating": 0
})

# Step 2: Fix negative values
bronze_orders_df = bronze_orders_df.withColumn(
    "total_amount",
    when(col("total_amount") < 0, abs(col("total_amount")))
    .otherwise(col("total_amount"))
).withColumn(
    "discount_applied",
    when(col("discount_applied") < 0, abs(col("discount_applied")))
    .otherwise(col("discount_applied"))
).withColumn(
    "feedback_rating",
    when(col("feedback_rating") < 0, abs(col("feedback_rating")))
    .otherwise(col("feedback_rating"))
)


In [0]:
bronze_orders_df.printSchema()
bronze_orders_df.select(
    "order_id",
    "customer_id",
    "restaurant_id",
    "status",
    "payment_mode",
    "total_amount",
    "discount_applied",
    "feedback_rating"
)
bronze_orders_df.display()

In [0]:
bronze_deliveries_df=bronze_deliveries_df.withColumn("delivery_id",col("delivery_id").cast("int"))\
    .withColumn("order_id",col("order_id").cast("int"))\
    .withColumn("delivery_person_id",col("delivery_person_id").cast("int"))\
    .withColumn("distance",col("distance").cast("string"))\
    .withColumn("delivery_time",col("delivery_time").cast("string"))\
    .withColumn("estimated_time",col("estimated_time").cast("string"))\
    .withColumn("delivery_fee",col("delivery_fee").cast("string"))\
    .withColumn("vehicle_type",col("vehicle_type").cast("string"))



In [0]:
bronze_deliveries_df.printSchema()

In [0]:
from pyspark.sql.functions import col, when, abs

bronze_deliveries_df = bronze_deliveries_df.replace(["null", "NULL", ""], None)

# cast + fill
bronze_deliveries_df = bronze_deliveries_df.withColumn("distance", expr("try_cast(distance as double)")) \
       .withColumn("estimated_time", expr("try_cast(estimated_time as int)")) \
       .withColumn("delivery_fee", expr("try_cast(delivery_fee as double)"))\
        .withColumn("delivery_time", expr("try_cast(delivery_time as double)"))

bronze_deliveries_df = bronze_deliveries_df.fillna({
    "delivery_status":"Unknown",
    "delivery_time": 0,
    "delivery_fee": 0,
    "estimated_time": 0,
    "vehicle_type": "Unknown",
    "distance":0
})

bronze_deliveries_df = bronze_deliveries_df.withColumn("delivery_fee",when(col("delivery_fee") < 0, abs(col("delivery_fee"))).otherwise(col("delivery_fee")))\
.withColumn("delivery_time",when(col("delivery_time") < 0, abs(col("delivery_time"))).otherwise(col("delivery_time")))\
    .withColumn("estimated_time",when(col("estimated_time") < 0, abs(col("estimated_time"))).otherwise(col("estimated_time")))\
    .withColumn("distance",when(col("distance") < 0, abs(col("distance"))).otherwise(col("distance")))


In [0]:
bronze_deliveries_df.printSchema()

In [0]:
bronze_deliveries_df.display()

In [0]:
bronze_customers_df.write.format("delta").mode("overwrite").saveAsTable("zomato_project.silver.silver_customers")
bronze_orders_df.write.format("delta").mode("overwrite").saveAsTable("zomato_project.silver.silver_orders")
bronze_deliveries_df.write.format("delta").mode("overwrite").saveAsTable("zomato_project.silver.silver_deliveries")
bronze_delivery_persons_df.write.format("delta").mode("overwrite").saveAsTable("zomato_project.silver.silver_delivery_persons")
bronze_restaurants_df.write.format("delta").mode("overwrite").saveAsTable("zomato_project.silver.silver_restaurants")

In [0]:
silver_customers_df=spark.read.table("zomato_project.silver.silver_customers")
silver_orders_df=spark.read.table("zomato_project.silver.silver_orders")
silver_deliveries_df=spark.read.table("zomato_project.silver.silver_deliveries")
silver_delivery_persons_df=spark.read.table("zomato_project.silver.silver_delivery_persons")
silver_restaurants_df=spark.read.table("zomato_project.silver.silver_restaurants")

silver_customers_df.display()
silver_orders_df.display()
silver_deliveries_df.display()
silver_delivery_persons_df.display()
silver_restaurants_df.display()